In [1]:
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Optional

PROJECT_ROOT = Path("/home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5")
sys.path.insert(0, str(PROJECT_ROOT))

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc, average_precision_score

import torch
from torch.utils.data import DataLoader

from models.doc_forgery_dataset_rgb_with_class_head_rtm import DocForgeryDatasetClassHead
from models.segformer_b2_mmseg_rgb_rtm_with_class_head.model_segformer_b2_mmseg_rgb_rtm_with_class_head import SegFormerBinaryClassificationRunner
from models.torchtools.torch_collate_segformer_b2_rgb_with_class_head_rtm import pad_collate_classification

from tensorboard.backend.event_processing import event_accumulator

/home/guests3/rma/.conda/envs/rtm/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/guests3/rma/.conda/envs/rtm/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/home/guests3/rma/.conda/envs/rtm/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/iss

In [40]:
# ============================================================
# Configuration
# ============================================================

EXP_NAME = "VAL_TEST/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm"  # change
EXP_NAME_TRAIN = "TRAIN/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/training_metric_plots"    # change

TRAIN_OUT_ROOT = Path("/home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo") / EXP_NAME_TRAIN
TRAIN_PLOTS_DIR = TRAIN_OUT_ROOT / "training_metric_plots"
TRAIN_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CKPT = (
    PROJECT_ROOT
    #/ "models"
    #/ "weights_segformer_b2_rtm_several_inputs"
    / "weights_mmseg_segformer_b2_rgb_with_class_head_rtm_corretoo"
    / "segformer_b2-mmseg-rgb-with-class-head"
    / "2026-05-14 16:23:42.185738-checkpoint9.pth"
)  # change

BACKBONE_CKPT = (
    PROJECT_ROOT
    / "models"
    / "RTMSegformer"
    / "configs"
    / "segformer"
    / "pretrain"
    / "mit_b2.pth"
)

VAL_IMG_DIR = "/media/general_storage6/rmastorage/datasets/RealTextManipulation/val_v2/JPEGImages"
VAL_MASK_DIR = "/media/general_storage6/rmastorage/datasets/RealTextManipulation/val_v2/SegmentationClass"

TEST_IMG_DIR = "/media/general_storage6/rmastorage/datasets/RealTextManipulation/test_v2/JPEGImages"
TEST_MASK_DIR = "/media/general_storage6/rmastorage/datasets/RealTextManipulation/test_v2/SegmentationClass"

RUN_DIR = (
    PROJECT_ROOT
    #/ "models"
    #/ "runs_segformer_b2_rtm_several_inputs"
    / "runs_segformer_b2_mmseg_rgb_with_class_head_rtm_corretoo"
    / "segformer_b2-mmseg-rgb-with-class-head"
)  # change

BATCH_SIZE = 8
NUM_WORKERS = 0
QF3 = 90

# 0-19 epochs
EPOCH_OFFSET = 0
EPOCH_MIN = 0
EPOCH_MAX = 20

# 20-49 epochs
#EPOCH_OFFSET = 0
#EPOCH_MIN = 20
#EPOCH_MAX = 50

# 50-99 epochs
#EPOCH_OFFSET = 0
#EPOCH_MIN = 50
#EPOCH_MAX = 100

# All epochs
#EPOCH_OFFSET = 0
#EPOCH_MIN = 0
#EPOCH_MAX = 50

DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")


print("TRAIN_PLOTS_DIR:", TRAIN_PLOTS_DIR)
print("DEVICE:", DEVICE)

TRAIN_PLOTS_DIR: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/TRAIN/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/training_metric_plots/training_metric_plots
DEVICE: cuda:3


In [41]:
# =============================================================================
# Output folders
# =============================================================================

OUT_ROOT = (
    PROJECT_ROOT
    / "models"
    / "eval_outputs_corretoo"
    / EXP_NAME
)  # change
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("OUT_ROOT:", OUT_ROOT)

OUT_ROOT: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/VAL_TEST/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm


In [4]:
# =============================================================================
# Load checkpoint
# =============================================================================

ckpt = torch.load(CKPT, map_location="cpu")

print("Checkpoint epoch:", ckpt.get("epoch", None))
print("Checkpoint best_val_metric:", ckpt.get("best_val_metric", None))
print("Checkpoint best_thr:", ckpt.get("best_thr", None))


Checkpoint epoch: 9
Checkpoint best_val_metric: 0.5207592248916626
Checkpoint best_thr: 0.28999999165534973


In [5]:
# =============================================================================
# Dataset and dataloader creation
# =============================================================================

def build_dataset(image_dir: str, mask_dir: str) -> DocForgeryDatasetClassHead:
    """Create a dataset with the same preprocessing used during evaluation."""
    
    dataset = DocForgeryDatasetClassHead(
        images_repo=[Path(image_dir)],
        masks_repo=[Path(mask_dir)],
        crop_size=(512, 512),
        grid_crop=True,
        seed=3,
        balance_crops=False,
        stride=512,
        use_augs=False,
        verbose_stats=True,
    )

    dataset.use_augs = False
    dataset.QF = QF3
    return dataset


def build_loader(dataset: DocForgeryDatasetClassHead) -> DataLoader:
    """Create a dataloader with the same classification collate used during training."""
    
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=pad_collate_classification,
    )


ds_val = build_dataset(VAL_IMG_DIR, VAL_MASK_DIR)
ds_test = build_dataset(TEST_IMG_DIR, TEST_MASK_DIR)

dl_val = build_loader(ds_val)
dl_test = build_loader(ds_test)

print("VAL windows:", len(ds_val))
print("TEST windows:", len(ds_test))


[INFO] docs=900 windows=7033 pos=1771 neg=5262 balance_crops=False p_pos_crop=0.5 stride=512
[INFO] docs=900 windows=7120 pos=1743 neg=5377 balance_crops=False p_pos_crop=0.5 stride=512
VAL windows: 7033
TEST windows: 7120


In [6]:
# ============================================================
# Load model
# ============================================================

runner = SegFormerBinaryClassificationRunner(
    load_path=CKPT,
)

runner.model.to(DEVICE)
runner.model.eval()

print("Model loaded on:", DEVICE)


[mit_b2] loaded backbone-only. missing=0 unexpected=0
... Loading trained weights
Done.
Model loaded on: cuda:3


In [7]:
# =============================================================================
# Basic metric helpers
# =============================================================================

TAMP_PREFIXES = {"cpmv", "insert", "splice", "edit", "inpaint", "cover"}
GOOD_PREFIXES = {"good"}


def group_from_stem(stem: str) -> str:
    """Map each image filename to its evaluation group."""
    
    prefix = stem.split("_")[0]

    if prefix in TAMP_PREFIXES:
        return "tamp"

    if prefix in GOOD_PREFIXES:
        return "good"


def get_doc_ids_from_batch(batch: dict) -> List[str]:
    """Extract document identifiers from a batch."""
    
    if "doc_id" in batch:
        return [str(doc_id) for doc_id in batch["doc_id"]]

    if "meta" in batch:
        return [str(meta["stem"]) for meta in batch["meta"]]

    raise KeyError("Batch must contain either 'doc_id' or 'meta' with document stems.")

In [8]:
# ============================================================
# Label and metric utilities
# ============================================================

def sigmoid_scalar(value: float) -> float:
    """Compute the sigmoid of a scalar value."""
    
    return float(1.0 / (1.0 + np.exp(-value)))


def compute_binary_metrics(
    y_true: List[bool],
    y_pred: List[bool],
    eps: float = 1e-8,
) -> dict:
    """Compute binary classification metrics from boolean targets and predictions."""
    
    tp = sum(pred and true for pred, true in zip(y_pred, y_true))
    fp = sum(pred and not true for pred, true in zip(y_pred, y_true))
    fn = sum((not pred) and true for pred, true in zip(y_pred, y_true))
    tn = sum((not pred) and (not true) for pred, true in zip(y_pred, y_true))

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    acc = (tp + tn) / (tp + tn + fp + fn + eps)

    tpr = tp / (tp + fn + eps)
    tnr = tn / (tn + fp + eps)
    bal_acc = 0.5 * (tpr + tnr)

    return {
        "n_docs": len(y_true),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "acc": acc,
        "bal_acc": bal_acc,
    }


In [9]:
@torch.no_grad()
def collect_document_logits_and_labels(
    model: SegFormerBinaryClassificationRunner,
    dataloader: DataLoader,
    device: torch.device,
) -> Tuple[Dict[str, List[float]], Dict[str, bool], Dict[str, str]]:
    """Collect crop/window-level classification logits and labels grouped by document."""

    doc_logits: Dict[str, List[float]] = {}
    doc_targets: Dict[str, bool] = {}
    doc_groups: Dict[str, str] = {}

    model.model.eval()

    for batch in dataloader:
        output = model.logits(
            batch=batch,
            device=device,
            loss_fn=None,
        )

        logits = output.logits.view(-1)
        labels = output.labels.view(-1).float()
        doc_ids = get_doc_ids_from_batch(batch)

        if len(doc_ids) != logits.numel():
            raise RuntimeError(
                f"Number of doc_ids ({len(doc_ids)}) does not match batch logits ({logits.numel()})."
            )

        for idx, doc_id in enumerate(doc_ids):
            doc_id = str(doc_id)

            logit_value = float(logits[idx].detach().cpu().item())
            target_value = bool(labels[idx].detach().cpu().item() >= 0.5)

            doc_logits.setdefault(doc_id, []).append(logit_value)
            doc_targets[doc_id] = bool(doc_targets.get(doc_id, False) or target_value)
            doc_groups[doc_id] = group_from_stem(doc_id)

    return doc_logits, doc_targets, doc_groups


In [10]:
def aggregate_document_logits(
    doc_logits: Dict[str, List[float]],
    aggregation: str = "max",
) -> Dict[str, float]:
    """Aggregate crop-level logits into one document-level logit."""

    doc_scores = {}

    for doc_id, logits in doc_logits.items():
        if aggregation == "max":
            doc_scores[doc_id] = float(max(logits))
        elif aggregation == "mean":
            doc_scores[doc_id] = float(sum(logits) / max(1, len(logits)))
        else:
            raise ValueError("aggregation must be either 'max' or 'mean'.")

    return doc_scores

In [11]:
def build_image_branch_document_table(
    doc_logits: Dict[str, List[float]],
    doc_targets: Dict[str, bool],
    doc_groups: Dict[str, str],
    aggregation: str = "max",
) -> pd.DataFrame:
    """Build a document-level table using only the image branch score."""

    doc_scores_logit = aggregate_document_logits(
        doc_logits=doc_logits,
        aggregation=aggregation,
    )

    rows = []

    for doc_id, logit_value in doc_scores_logit.items():
        rows.append(
            {
                "doc_id": doc_id,
                "group": doc_groups.get(doc_id, "unknown"),
                "target": bool(doc_targets.get(doc_id, False)),
                "logit_cls": float(logit_value),
                "score_cls": sigmoid_scalar(logit_value),
            }
        )

    return pd.DataFrame(rows)

In [12]:
def build_document_level_report(
    df_test: pd.DataFrame,
    threshold: float,
) -> pd.DataFrame:
    """Build document-level metrics for tampered documents and all documents."""

    rows = []

    for split_name, group_name in [("Tampered", "tamp"), ("All", None)]:
        metrics = evaluate_scores_at_threshold(
            df=df_test,
            score_column="score_cls",
            threshold=threshold,
            group_name=group_name,
        )

        row = {
            "split": split_name,
            "threshold": float(threshold),
        }
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows)

In [13]:
def build_good_only_report(
    df_test: pd.DataFrame,
    threshold: float,
) -> pd.DataFrame:
    """Build good-document-only metrics."""

    df_good = df_test[df_test["group"] == "good"].copy()

    scores = df_good["score_cls"].values.astype(float)
    predictions = scores >= threshold

    fp_good = int(predictions.sum())
    tn_good = int((~predictions).sum())

    total_good = fp_good + tn_good
    eps = 1e-8

    return pd.DataFrame(
        [
            {
                "threshold": float(threshold),
                "n_good": total_good,
                "fp_good": fp_good,
                "tn_good": tn_good,
                "fpr_good": float(fp_good / (total_good + eps)),
                "specificity_good": float(tn_good / (total_good + eps)),
            }
        ]
    )

In [14]:
# ============================================================
# AUC and threshold-dependent evaluation table
# ============================================================

def recall_at_fpr(
    y_true: np.ndarray,
    scores: np.ndarray,
    target_fpr: float,
) -> float:
    """Compute the maximum recall obtained with FPR below a target value."""
    
    if len(np.unique(y_true)) < 2:
        return float("nan")

    fpr, tpr, _ = roc_curve(y_true, scores)

    valid = fpr <= target_fpr
    if not np.any(valid):
        return 0.0

    return float(np.max(tpr[valid]))


def compute_auc_metrics(
    y_true: np.ndarray,
    scores: np.ndarray,
) -> dict:
    """Compute threshold-independent document-level metrics."""
    
    if len(np.unique(y_true)) < 2:
        return {
            "ROC_AUC": float("nan"),
            "PR_AUC": float("nan"),
            "Recall@FPR=1%": float("nan"),
            "Recall@FPR=5%": float("nan"),
            "Recall@FPR=10%": float("nan"),
        }

    fpr, tpr, _ = roc_curve(y_true, scores)

    return {
        "ROC_AUC": float(auc(fpr, tpr)),
        "PR_AUC": float(average_precision_score(y_true, scores)),
        "Recall@FPR=1%": recall_at_fpr(y_true, scores, 0.01),
        "Recall@FPR=5%": recall_at_fpr(y_true, scores, 0.05),
        "Recall@FPR=10%": recall_at_fpr(y_true, scores, 0.10),
    }


def compute_threshold_metrics_from_scores(
    y_true: np.ndarray,
    scores: np.ndarray,
    threshold: float,
    eps: float = 1e-8,
) -> dict:
    """Compute threshold-dependent binary metrics from continuous scores."""
    
    y_true = y_true.astype(bool)
    y_pred = scores >= threshold

    tp = np.logical_and(y_pred, y_true).sum()
    fp = np.logical_and(y_pred, ~y_true).sum()
    fn = np.logical_and(~y_pred, y_true).sum()
    tn = np.logical_and(~y_pred, ~y_true).sum()

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    acc = (tp + tn) / (tp + tn + fp + fn + eps)

    fpr = fp / (fp + tn + eps)
    specificity = tn / (tn + fp + eps)

    return {
        "P@thr": float(precision),
        "R@thr": float(recall),
        "F1@thr": float(f1),
        "Acc@thr": float(acc),
        "FPR@thr": float(fpr),
        "Spec@thr": float(specificity),
    }

In [15]:
def build_auc_report(
    df_test: pd.DataFrame,
    threshold: float,
) -> pd.DataFrame:
    """Build AUC and threshold-dependent metrics for the image branch."""

    y_true = df_test["target"].astype(bool).values
    scores = df_test["score_cls"].values.astype(float)

    row = {
        "threshold": float(threshold),
    }

    row.update(
        compute_auc_metrics(
            y_true=y_true,
            scores=scores,
        )
    )

    row.update(
        compute_threshold_metrics_from_scores(
            y_true=y_true,
            scores=scores,
            threshold=threshold,
        )
    )

    return pd.DataFrame([row])

In [16]:
# ============================================================
# Threshold selection
# ============================================================

def default_threshold_grid() -> torch.Tensor:
    """Create the same threshold grid used in the classification training validation."""
    
    return torch.linspace(0.01, 0.99, 99)


def find_best_threshold_from_scores(
    scores: np.ndarray,
    targets: np.ndarray,
    thresholds: Optional[np.ndarray] = None,
    eps: float = 1e-8,
) -> dict:
    """Select the threshold that maximizes F1 for a given score vector."""
    
    if thresholds is None:
        thresholds = default_threshold_grid().cpu().numpy()

    best = {
        "threshold": 0.5,
        "precision": 0.0,
        "recall": 0.0,
        "f1": -1.0,
    }

    targets_bool = targets.astype(bool)

    for threshold in thresholds:
        predictions = scores >= threshold

        tp = np.logical_and(predictions, targets_bool).sum()
        fp = np.logical_and(predictions, ~targets_bool).sum()
        fn = np.logical_and(~predictions, targets_bool).sum()

        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)

        if f1 > best["f1"]:
            best = {
                "threshold": float(threshold),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }

    return best


In [17]:
# ============================================================
# Evaluation
# ============================================================

def evaluate_scores_at_threshold(
    df: pd.DataFrame,
    score_column: str,
    threshold: float,
    group_name: Optional[str] = None,
) -> dict:
    """Evaluate one score column at a fixed threshold."""
    
    if group_name is None:
        df_eval = df.copy()
    else:
        df_eval = df[df["group"] == group_name].copy()

    y_true = df_eval["target"].astype(bool).tolist()
    y_pred = (df_eval[score_column].values >= threshold).tolist()

    return compute_binary_metrics(
        y_true=y_true,
        y_pred=y_pred,
    )

In [18]:
# ============================================================
# Validation threshold selection and test evaluation
# ============================================================

MODEL_TAG = "segformer_b2_binary_classification_image_only"

print("Collecting validation document scores...")

val_doc_logits, val_doc_targets, val_doc_groups = collect_document_logits_and_labels(
    model=runner,
    dataloader=dl_val,
    device=DEVICE,
)

df_val_doc = build_image_branch_document_table(
    doc_logits=val_doc_logits,
    doc_targets=val_doc_targets,
    doc_groups=val_doc_groups,
    aggregation="max",
)

print("Validation documents:", len(df_val_doc))

targets_val = df_val_doc["target"].astype(bool).values

best_cls = find_best_threshold_from_scores(
    scores=df_val_doc["score_cls"].values,
    targets=targets_val,
)

thr_cls_doc = best_cls["threshold"]
ckpt_thr = ckpt.get("best_thr", None)

thresholds_df = pd.DataFrame(
    [
        {
            "method": "classification_branch_validation_selected",
            "threshold": thr_cls_doc,
            "validation_f1": best_cls["f1"],
            "validation_precision": best_cls["precision"],
            "validation_recall": best_cls["recall"],
        },
        {
            "method": "checkpoint_best_thr",
            "threshold": float(ckpt_thr) if ckpt_thr is not None else np.nan,
            "validation_f1": np.nan,
            "validation_precision": np.nan,
            "validation_recall": np.nan,
        },
    ]
)

print("\nValidation-selected threshold")
print("CLS threshold:", thr_cls_doc)
print("Validation F1:", best_cls["f1"])
print("Validation precision:", best_cls["precision"])
print("Validation recall:", best_cls["recall"])
print("Checkpoint best_thr:", ckpt_thr)

print("\nCollecting test document scores...")

test_doc_logits, test_doc_targets, test_doc_groups = collect_document_logits_and_labels(
    model=runner,
    dataloader=dl_test,
    device=DEVICE,
)

df_test_doc = build_image_branch_document_table(
    doc_logits=test_doc_logits,
    doc_targets=test_doc_targets,
    doc_groups=test_doc_groups,
    aggregation="max",
)

print("Test documents:", len(df_test_doc))

report_df = build_document_level_report(
    df_test=df_test_doc,
    threshold=thr_cls_doc,
)

good_report_df = build_good_only_report(
    df_test=df_test_doc,
    threshold=thr_cls_doc,
)

auc_report_df = build_auc_report(
    df_test=df_test_doc,
    threshold=thr_cls_doc,
)

print("\nDocument-level metrics:")
print(report_df)

print("\nGood-document-only metrics:")
print(good_report_df)

print("\nAUC metrics:")
print(auc_report_df)


Validation documents: 900

Validation-selected threshold
CLS threshold: 0.17999999225139618
Validation F1: 0.858814918251523
Validation precision: 0.7653194263263974
Validation recall: 0.9783333333170278
Checkpoint best_thr: 0.28999999165534973

Test documents: 900

Document-level metrics:
      split  threshold  n_docs   tp   fp  fn   tn  precision    recall  \
0  Tampered       0.18     600  583    0  17    0   1.000000  0.971667   
1       All       0.18     900  583  193  17  107   0.751289  0.971667   

         f1       acc   bal_acc  
0  0.985630  0.971667  0.485833  
1  0.847384  0.766667  0.664167  

Good-document-only metrics:
   threshold  n_good  fp_good  tn_good  fpr_good  specificity_good
0       0.18     300      193      107  0.643333          0.356667

AUC metrics:
   threshold   ROC_AUC    PR_AUC  Recall@FPR=1%  Recall@FPR=5%  \
0       0.18  0.796117  0.865034       0.106667       0.201667   

   Recall@FPR=10%     P@thr     R@thr    F1@thr   Acc@thr   FPR@thr  Spec@

In [19]:
def save_csv_and_png_table(
    df: pd.DataFrame,
    out_dir: Path,
    csv_name: str,
    png_name: str,
    decimals: int = 4,
    title: Optional[str] = None,
) -> Tuple[Path, Path]:
    """Save a DataFrame as both a CSV file and a PNG table."""
    
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / csv_name
    df.to_csv(csv_path, index=False)

    df_fmt = df.copy()

    for column in df_fmt.columns:
        if pd.api.types.is_numeric_dtype(df_fmt[column]):
            df_fmt[column] = df_fmt[column].map(lambda value: f"{value:.{decimals}f}")

    n_rows, n_cols = df_fmt.shape

    fig_width = min(22, 1.4 + 1.1 * n_cols)
    fig_height = max(3.0, 0.8 + 0.45 * n_rows)

    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height),
        dpi=220,
    )

    ax.axis("off")

    table = ax.table(
        cellText=df_fmt.values,
        colLabels=df_fmt.columns.tolist(),
        cellLoc="center",
        colLoc="center",
        loc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.5)

    for col_idx in range(n_cols):
        cell = table[0, col_idx]
        cell.set_text_props(weight="bold")
        cell.set_linewidth(1.0)

    for row_idx in range(1, n_rows + 1):
        for col_idx in range(n_cols):
            table[row_idx, col_idx].set_linewidth(0.8)

    if title is not None:
        ax.set_title(title, fontsize=12, pad=12)

    png_path = out_dir / png_name

    plt.tight_layout()
    plt.savefig(
        png_path,
        bbox_inches="tight",
    )
    plt.close(fig)

    return csv_path, png_path

In [20]:
# ================================================================
# Save document-level reports and validation-selected thresholds
# ================================================================

df_val_doc.to_csv(
    OUT_ROOT / f"validation_document_scores_{MODEL_TAG}.csv",
    index=False,
)

df_test_doc.to_csv(
    OUT_ROOT / f"test_document_scores_{MODEL_TAG}.csv",
    index=False,
)

thresholds_df.to_csv(
    OUT_ROOT / f"validation_selected_threshold_{MODEL_TAG}.csv",
    index=False,
)

csv_path, png_path = save_csv_and_png_table(
    df=report_df,
    out_dir=OUT_ROOT,
    csv_name=f"test_document_level_metrics_{MODEL_TAG}.csv",
    png_name=f"test_document_level_metrics_{MODEL_TAG}.png",
    title="TEST — Image branch only document-level metrics",
)

csv_path_good, png_path_good = save_csv_and_png_table(
    df=good_report_df,
    out_dir=OUT_ROOT,
    csv_name=f"test_good_documents_only_{MODEL_TAG}.csv",
    png_name=f"test_good_documents_only_{MODEL_TAG}.png",
    title="TEST — Image branch only good-document metrics",
)

csv_path_auc, png_path_auc = save_csv_and_png_table(
    df=auc_report_df,
    out_dir=OUT_ROOT,
    csv_name=f"test_auc_metrics_{MODEL_TAG}.csv",
    png_name=f"test_auc_metrics_{MODEL_TAG}.png",
    title="TEST — Image branch only AUC metrics",
)

print("Saved:", csv_path)
print("Saved:", png_path)
print("Saved:", csv_path_good)
print("Saved:", png_path_good)
print("Saved:", csv_path_auc)
print("Saved:", png_path_auc)

Saved: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/VAL_TEST/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/test_document_level_metrics_segformer_b2_binary_classification_image_only.csv
Saved: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/VAL_TEST/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/test_document_level_metrics_segformer_b2_binary_classification_image_only.png
Saved: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/VAL_TEST/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/test_good_documents_only_segformer_b2_binary_classification_image_only.csv
Saved: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/VAL_TEST/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/test_good_documents_only_segformer_b2_binary_classification_image_only.png
Saved: /home/guests3/rma/tese/

In [42]:
# ============================================================
# TensorBoard scalar loading
# ============================================================

def load_tensorboard_scalars(run_dir: Path) -> pd.DataFrame:
    """Load all scalar values from TensorBoard event files into a DataFrame."""
    
    event_files = sorted(run_dir.glob("**/events.out.tfevents.*"))

    rows = []

    for event_file in event_files:
        accumulator = event_accumulator.EventAccumulator(str(event_file))
        accumulator.Reload()

        scalar_tags = accumulator.Tags().get("scalars", [])

        for tag in scalar_tags:
            for event in accumulator.Scalars(tag):
                rows.append(
                    {
                        "file": str(event_file),
                        "tag": tag,
                        "step": event.step,
                        "value": event.value,
                    }
                )

    return pd.DataFrame(rows)

In [43]:
# ============================================================
# TensorBoard step-to-epoch mapping utilities
# ============================================================

def pick_anchor_tag(
    tag_counts: pd.Series,
    preferred_tags: Optional[List[str]] = None,
    min_points: int = 5,
) -> str:
    """Select a scalar tag used to map TensorBoard steps to epoch indices."""
    
    if preferred_tags is None:
        preferred_tags = [
            "Validation Loss",
            "Validation IMG Precision",
            "Validation IMG Recall",
            "Validation IMG BalAcc",
            "Validation IMG F1",
            "Validation IMG Acc",
            "Training Loss",
            "Training IMG Acc",
            "Training IMG BalAcc",
            "Training IMG F1",
            "Training IMG Precision",
            "Training IMG Recall",
        ]

    for tag in preferred_tags:
        if tag in tag_counts.index and tag_counts[tag] >= min_points:
            return tag

    candidates = tag_counts[tag_counts >= min_points]

    if len(candidates) == 0:
        return str(tag_counts.index[0])

    return str(candidates.index[0])


def add_epoch_column(
    df_scalars: pd.DataFrame,
    epoch_offset: int = 0,
) -> pd.DataFrame:
    """Map TensorBoard steps to epoch indices using a stable anchor tag."""
    
    if df_scalars.empty:
        return df_scalars.copy()

    tag_counts = df_scalars.groupby("tag")["step"].nunique().sort_values(ascending=False)
    anchor_tag = pick_anchor_tag(tag_counts)

    anchor_steps = sorted(df_scalars[df_scalars["tag"] == anchor_tag]["step"].unique())
    step_to_epoch = {step: idx for idx, step in enumerate(anchor_steps)}

    df_epochs = df_scalars.copy()
    df_epochs["epoch"] = df_epochs["step"].map(step_to_epoch)
    df_epochs = df_epochs.dropna(subset=["epoch"]).copy()
    df_epochs["epoch"] = df_epochs["epoch"].astype(int) + epoch_offset

    print("Anchor tag:", anchor_tag)
    print("Number of anchor steps:", len(anchor_steps))
    print("First steps:", anchor_steps[:10])

    return df_epochs

In [44]:
# ============================================================
# Epoch range filtering
# ============================================================

def filter_epoch_range(
    df_epochs: pd.DataFrame,
    epoch_min: Optional[int] = None,
    epoch_max: Optional[int] = None,
) -> pd.DataFrame:
    """Filter TensorBoard scalar data by epoch range."""

    df_filtered = df_epochs.copy()

    if epoch_min is not None:
        df_filtered = df_filtered[df_filtered["epoch"] >= epoch_min]

    if epoch_max is not None:
        df_filtered = df_filtered[df_filtered["epoch"] < epoch_max]

    return df_filtered.copy()

In [45]:
# ============================================================
# Metric selection utilities
# ============================================================

def select_available_metrics(
    df_epochs: pd.DataFrame,
    metrics_of_interest: List[str],
) -> List[str]:
    """Return the requested TensorBoard tags that are available in the run."""
    
    available_tags = set(df_epochs["tag"].unique())

    return [
        tag
        for tag in metrics_of_interest
        if tag in available_tags
    ]

In [46]:
# ============================================================
# Save TensorBoard training plots and per-epoch CSV
# ============================================================

def save_training_metric_plots(
    df_epochs: pd.DataFrame,
    output_dir: Path,
    metrics_of_interest: Optional[List[str]] = None,
    epoch_min: Optional[int] = None,
    epoch_max: Optional[int] = None,
) -> pd.DataFrame:
    """Save one line plot per selected TensorBoard metric and export a CSV summary."""
    
    output_dir.mkdir(parents=True, exist_ok=True)

    df_epochs = filter_epoch_range(
        df_epochs=df_epochs,
        epoch_min=epoch_min,
        epoch_max=epoch_max,
    )
    
    if df_epochs.empty:
        print(f"No TensorBoard scalar rows found in epoch range [{epoch_min}, {epoch_max}).")
        return pd.DataFrame()

    if metrics_of_interest is None:
        metrics_of_interest = [
            "Training Loss",
            "Training UseThr",
            "Training Precision@UseThr",
            "Training Recall@UseThr",
            "Training F1@UseThr",
            "Training Accuracy@UseThr",
            "Validation Loss",
            "Validation UseThr",
            "Validation Precision@UseThr",
            "Validation Recall@UseThr",
            "Validation F1@UseThr",
            "Validation Accuracy@UseThr",
            "Validation BestF1",
            "Validation BestThreshold",
            "Validation_BAL Loss",
            "Validation_BAL Precision@UseThr",
            "Validation_BAL Recall@UseThr",
            "Validation_BAL F1@UseThr",
            "Validation_BAL Accuracy@UseThr",
            "Validation_BAL BestF1",
            "Validation_BAL BestThreshold",
            "Epoch Duration",
        ]

    available_metrics = select_available_metrics(
        df_epochs=df_epochs,
        metrics_of_interest=metrics_of_interest,
    )

    print("Available training metrics:", available_metrics)

    for tag in available_metrics:
        metric_df = df_epochs[df_epochs["tag"] == tag].sort_values("epoch")
    
        plt.figure(figsize=(8, 4))
        plt.plot(metric_df["epoch"], metric_df["value"], marker="o")

        # the classification training code saves the best checkpoint using Validation BestF1 (Best image-level binary classification F1)
        # therefore, only this plot receives the fixed-epoch and best-epoch highlights
        if tag == "Validation BestF1":
            # fixed epoch highlight (red)
            highlight_epoch = 35
            point = metric_df[metric_df["epoch"] == highlight_epoch]
        
            if not point.empty:
                highlight_value = point["value"].iloc[0]
        
                plt.plot(
                    highlight_epoch,
                    highlight_value,
                    marker="o",
                    color="red",
                    markersize=8,
                    linestyle="None",
                    label=f"Epoch {highlight_epoch}",
                    zorder=5,
                )
        
            # best value in the plot (green highlight)
            best_idx = metric_df["value"].idxmax()
        
            best_epoch = metric_df.loc[best_idx, "epoch"]
            best_value = metric_df.loc[best_idx, "value"]
        
            plt.plot(
                best_epoch,
                best_value,
                marker="o",
                color="green",
                markersize=8,
                linestyle="None",
                label=f"Best Epoch {best_epoch}",
                zorder=6,
            )
        
            plt.legend()
    
        plt.title(tag)
        plt.xlabel("Epoch")
        plt.ylabel("Value")
        plt.grid(True)

        if epoch_min is not None or epoch_max is not None:
            plt.xlim(
                left=epoch_min if epoch_min is not None else df_epochs["epoch"].min(),
                right=(epoch_max - 1) if epoch_max is not None else df_epochs["epoch"].max(),
            )

        safe_name = (
            tag.replace(" ", "_")
            .replace("/", "_")
            .replace("@", "at")
        )

        plt.savefig(
            output_dir / f"{safe_name}.png",
            dpi=200,
            bbox_inches="tight",
        )
        plt.close()

    summary = (
        df_epochs.pivot_table(
            index="epoch",
            columns="tag",
            values="value",
            aggfunc="last",
        )
        .sort_index()
    )

    summary_path = output_dir.parent / "training_metrics_per_epoch.csv"
    summary.to_csv(summary_path)

    print("Saved training plots to:", output_dir)
    print("Saved training CSV to:", summary_path)

    return summary


In [47]:
# ============================================================
# Export TensorBoard training curves
# ============================================================

def export_training_curves(
    run_dir: Path,
    output_dir: Path,
    epoch_offset: int = 0,
    epoch_min: Optional[int] = None,
    epoch_max: Optional[int] = None,
) -> pd.DataFrame:
    """Load TensorBoard scalars, create training plots, and save the per-epoch CSV."""
    
    df_scalars = load_tensorboard_scalars(run_dir)

    if df_scalars.empty:
        print("No TensorBoard scalar events found in:", run_dir)
        return pd.DataFrame()

    print("TensorBoard scalar rows:", len(df_scalars))
    print("Unique scalar tags:", df_scalars["tag"].nunique())

    df_epochs = add_epoch_column(
        df_scalars=df_scalars,
        epoch_offset=epoch_offset,
    )

    return save_training_metric_plots(
        df_epochs=df_epochs,
        output_dir=output_dir,
        epoch_min=epoch_min,
        epoch_max=epoch_max,
    )

In [48]:
export_training_curves(
    run_dir=RUN_DIR,
    output_dir=TRAIN_PLOTS_DIR,
    epoch_offset=EPOCH_OFFSET,
    epoch_min=EPOCH_MIN,
    epoch_max=EPOCH_MAX,
)

TensorBoard scalar rows: 11300
Unique scalar tags: 23
Anchor tag: Validation Loss
Number of anchor steps: 50
First steps: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Available training metrics: ['Training Loss', 'Training UseThr', 'Training Precision@UseThr', 'Training Recall@UseThr', 'Training F1@UseThr', 'Training Accuracy@UseThr', 'Validation Loss', 'Validation UseThr', 'Validation Precision@UseThr', 'Validation Recall@UseThr', 'Validation F1@UseThr', 'Validation Accuracy@UseThr', 'Validation BestF1', 'Validation BestThreshold', 'Validation_BAL Loss', 'Validation_BAL Precision@UseThr', 'Validation_BAL Recall@UseThr', 'Validation_BAL F1@UseThr', 'Validation_BAL Accuracy@UseThr', 'Validation_BAL BestF1', 'Validation_BAL BestThreshold', 'Epoch Duration']
Saved training plots to: /home/guests3/rma/tese/codigo_tese/detect_forgery_documents/opcao_5/models/eval_outputs_corretoo/TRAIN/50 epochs/segformer_b2_mmseg_rgb_with_class_head_rtm/training_metric_plots/training_metric_plots
Saved training CSV to: 

tag,Epoch Duration,Running Training Loss,Training Accuracy@UseThr,Training F1@UseThr,Training Loss,Training Precision@UseThr,Training Recall@UseThr,Training UseThr,Validation Accuracy@UseThr,Validation BestF1,...,Validation Precision@UseThr,Validation Recall@UseThr,Validation UseThr,Validation_BAL Accuracy@UseThr,Validation_BAL BestF1,Validation_BAL BestThreshold,Validation_BAL F1@UseThr,Validation_BAL Loss,Validation_BAL Precision@UseThr,Validation_BAL Recall@UseThr
epoch,,,,,,,,,,,,,,,,,,,,,
0,429.266388,NaN,0.635521,0.639943,0.635516,0.626302,0.654192,0.50,0.608844,0.497302,...,0.366558,0.760023,0.50,0.655339,0.699802,0.34,0.687226,0.607419,0.620457,0.770098
1,425.737854,NaN,0.659078,0.637218,0.581533,0.699145,0.585369,0.57,0.678089,0.502468,...,0.409342,0.628458,0.57,0.658894,0.716837,0.33,0.647258,0.599090,0.676814,0.620175
2,425.612915,NaN,0.687015,0.710162,0.564005,0.661914,0.765997,0.48,0.617944,0.513158,...,0.376015,0.784303,0.48,0.674677,0.718070,0.31,0.708386,0.591077,0.646429,0.783479
3,425.642456,NaN,0.700876,0.708108,0.549330,0.683534,0.734516,0.51,0.668136,0.507103,...,0.403892,0.667984,0.51,0.667852,0.715122,0.36,0.670429,0.600507,0.664801,0.676153
4,426.742493,NaN,0.713085,0.722858,0.539048,0.701023,0.746098,0.51,0.627328,0.510837,...,0.378154,0.744777,0.51,0.665292,0.717352,0.32,0.687798,0.601145,0.638827,0.744901
5,425.891632,0.760222,0.724504,0.734349,0.517899,0.711582,0.758621,0.51,0.713920,0.506199,...,0.445401,0.555054,0.51,0.661738,0.728506,0.13,0.627291,0.658388,0.711443,0.560941
6,426.786255,NaN,0.742172,0.752584,0.490551,0.723946,0.783582,0.51,0.701692,0.501851,...,0.431903,0.585545,0.51,0.660173,0.723060,0.14,0.631968,0.657135,0.697959,0.577378
7,426.616882,NaN,0.759049,0.768828,0.480185,0.737963,0.802387,0.51,0.665008,0.513069,...,0.403878,0.693958,0.51,0.672117,0.722385,0.15,0.678741,0.669910,0.671444,0.686197
8,426.891327,NaN,0.780882,0.787045,0.453918,0.761892,0.813917,0.51,0.679511,0.509267,...,0.413161,0.648786,0.51,0.664012,0.717655,0.17,0.657883,0.732962,0.675788,0.640903
